# AgentRx preview notebook

This notebook is prepared for the **gated** Hugging Face dataset **microsoft/AgentRx**.

It will work **after**:
1. you request/obtain dataset access on Hugging Face, and  
2. you authenticate locally with `huggingface-cli login` or `notebook_login()`.

**Source and access note**  
The dataset card says AgentRx is gated and requires accepting terms to access its files. It exposes two domain splits, `tau_retail` and `magentic_one`, with fields such as `trajectory_id`, `failure_summary`, `failures`, `root_cause`, and `num_failures`. citeturn513868view3turn720793view3

In [ ]:
!pip -q install datasets pandas huggingface_hub

In [1]:
import json
import ast
from pathlib import Path
from typing import Any
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 200)

def truncate(text: Any, n: int = 300) -> str:
    if text is None:
        return ""
    s = str(text)
    return s if len(s) <= n else s[:n] + " …"

def maybe_parse_json(x: Any):
    if isinstance(x, (dict, list)):
        return x
    if not isinstance(x, str):
        return x
    x = x.strip()
    if not x:
        return x
    for fn in (json.loads, ast.literal_eval):
        try:
            return fn(x)
        except Exception:
            pass
    return x

def first_dialogue_excerpt(obj: Any, max_items: int = 4) -> str:
    parsed = maybe_parse_json(obj)
    if isinstance(parsed, list):
        parts = []
        for item in parsed[:max_items]:
            if isinstance(item, dict):
                role = item.get("role") or item.get("userType") or item.get("type") or "item"
                content = item.get("content") or item.get("text") or item.get("message") or item.get("action") or item.get("thought") or item
                parts.append(f"{role}: {truncate(content, 180)}")
            else:
                parts.append(truncate(item, 180))
        return "\n".join(parts)
    if isinstance(parsed, dict):
        return truncate(json.dumps(parsed, indent=2), 500)
    return truncate(parsed, 500)

def choose_first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None

In [ ]:
# Run this only if you have not already authenticated:
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:

from huggingface_hub import login
import os

hf_token = os.environ["HUGGING_FACE_TOKEN"]
login(hf_token)

from datasets import load_dataset

split_name = "tau_retail"   # or "magentic_one"
ds = load_dataset("microsoft/AgentRx", split=split_name)
df = ds.to_pandas()
print(df.shape)
display(df.head())

tau_retail.jsonl: 0.00B [00:00, ?B/s]

magentic_one.jsonl: 0.00B [00:00, ?B/s]

Generating tau_retail split:   0%|          | 0/29 [00:00<?, ? examples/s]

Generating magentic_one split:   0%|          | 0/44 [00:00<?, ? examples/s]

(29, 7)


,trajectory_id,failure_summary,failures,num_failures,root_cause,root_cause_failure_id,root_cause_reason
0,2,The agent did not correctly count the number of available t-shirts from the tool call result.,"[{'failure_id': '1', 'step_number': 3, 'step_reason': 'At step 3, the assistant agent did not authenticate user information before proceeding to provide information about available t-shirts', 'fai...",2,"{'failure_id': '2', 'reason_for_root_cause': 'The assistant finally did authenticate before providing user specific information. The incorrect count does not correspond with ground truth output.'}",2,The assistant finally did authenticate before providing user specific information. The incorrect count does not correspond with ground truth output.
1,3,The assistant did not correctly count the number of available t-shirts from the tool call result.,"[{'failure_id': '3', 'step_number': 3, 'step_reason': 'At step 3, The assistant agent did not authenticate user information before proceeding to provide information about available t-shirts', 'fai...",2,"{'failure_id': '4', 'reason_for_root_cause': 'The assistant finally did authenticate before providing user specific information. The incorrect count does not correspond with ground truth output.'}",4,The assistant finally did authenticate before providing user specific information. The incorrect count does not correspond with ground truth output.
2,4,Assistant misinterpreted the tool output and incorrectly counted the number of available t-shirts.,"[{'failure_id': '5', 'step_number': 15, 'step_reason': 'At step 15, the assistant agent incorrectly counted the number of available t-shirts.', 'failure_category': 'Misinterpretation of Tool Outpu...",1,"{'failure_id': '5', 'reason_for_root_cause': 'The incorrect count does not correspond with ground truth output.'}",5,The incorrect count does not correspond with ground truth output.
3,12,Assistant did not ask user for confirmation and payment method before initiating return.,"[{'failure_id': '6', 'step_number': 19, 'step_reason': 'At step 19, the assistant agent did not ask user for confirmation and payment method before initiating return.', 'failure_category': 'Instru...",1,"{'failure_id': '6', 'reason_for_root_cause': 'The agent did not recover from this error.'}",6,The agent did not recover from this error.
4,20,Assistant mistook two of the orders that were processed as delivered.,"[{'failure_id': '7', 'step_number': 21, 'step_reason': 'At step 21, the assistant mistook two of the orders which were processed as delivered', 'failure_category': 'Misinterpretation of Tool Outpu...",4,"{'failure_id': '7', 'reason_for_root_cause': 'The assistant did not recover from misinterpretation error, which led to subsequent failures in the trajectory.'}",7,"The assistant did not recover from misinterpretation error, which led to subsequent failures in the trajectory."


In [ ]:
preview_cols = [c for c in ["trajectory_id", "failure_summary", "root_cause_failure_id", "num_failures"] if c in df.columns]
display(df[preview_cols].head(10))

In [ ]:
examples = []
for _, row in df.head(5).iterrows():
    failures = maybe_parse_json(row.get("failures"))
    root_cause = maybe_parse_json(row.get("root_cause"))
    examples.append({
        "trajectory_id": row.get("trajectory_id"),
        "num_failures": row.get("num_failures"),
        "failure_summary": truncate(row.get("failure_summary"), 250),
        "first_failure_excerpt": truncate(failures[0] if isinstance(failures, list) and failures else None, 250),
        "root_cause_excerpt": truncate(root_cause, 250),
    })
example_df = pd.DataFrame(examples)
display(example_df)